In [ ]:
import pandas as pd
Data=pd.read_csv('/content/CTrain.csv')
Test=pd.read_csv('/content/CEV_contest_test_dataset.csv')

In [ ]:
b=Test[["MEMORY","ORIENT","JUDGMENT","COMMUN","HOMEHOBB","PERSCARE"]]
b.head()

In [ ]:
Test["SYMPTOMS"]=Test["MEMORY"] + Test["ORIENT"] + Test["JUDGMENT"] + Test["COMMUN"] + Test["HOMEHOBB"]
Test["SYMPTOMS"]=Test["SYMPTOMS"]/5
#Test.drop("SYMTOMS",axis=1,inplace=True)
Test.head()

In [ ]:
Test.SYMPTOMS.replace([ 2.4, 2.8 ], 3.0, inplace=True) #toplamda 2 olmalı (2)
Test.SYMPTOMS.replace([ 1.4, 1.6, 1.8, 2. , 2.2,],2.0, inplace=True) #toplamda 20 olmalı ( 20 küsür gibi  yapıyor)
Test.SYMPTOMS.replace([ 0.7, 0.8, 0.9, 1. , 1.1, 1.2,
       1.3,] ,1.0, inplace=True) #toplamda 70'e yakın olmalı (70 gibi yapıyor)
Test.SYMPTOMS.replace([0.2,0.3, 0.4, 0.5, 0.6] ,0.5, inplace=True) #toplamda 955'e yakın olmalı 
Test.SYMPTOMS.replace([0. , 0.1] ,0.0, inplace=True) #toplamda 3475'e uyakın olmaası gerek 0.2 de buraya eklenebilir 

In [ ]:
Test['SYMPTOMS'].value_counts()

In [ ]:
Test=Test.drop(["MEMORY","ORIENT","JUDGMENT","COMMUN","HOMEHOBB"],axis=1,inplace=False)
Test.corr().style.background_gradient(cmap='coolwarm')

In [ ]:
Data["SEX"].replace({1: 0, 2: 1}, inplace=True)
Test["SEX"].replace({1: 0, 2: 1}, inplace=True)
Data['SEX'],Test['SEX']

In [ ]:
categorical_cols = ['NACCNE4S','INDEPEND'] 

#import pandas as pd
df = pd.get_dummies(Data[categorical_cols], columns = categorical_cols)
dft = pd.get_dummies(Test[categorical_cols], columns = categorical_cols)

In [ ]:
Data = Data.join(df)
Test=Test.join(dft)

In [ ]:
Data=Data.drop(columns=['NACCNE4S', 'INDEPEND','Unnamed: 0'], axis=1)
Test=Test.drop(columns=['NACCNE4S', 'INDEPEND','Unnamed: 0'], axis=1)
Data.head(),Test.head()

In [ ]:
Data.head().T,Test.head().T

In [ ]:
Y_train=Data['SYMPTOMS']
Y_test=Test['SYMPTOMS']                
Data=Data.drop(columns=['SYMPTOMS'],axis=1)
Test=Test.drop(columns=['SYMPTOMS'],axis=1)

In [ ]:
Data.head().T

In [ ]:
Test.head().T

In [ ]:
Y_train.value_counts(),Y_test.value_counts()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
PredictorScaler=MinMaxScaler()
TargetVarScaler=MinMaxScaler()
 
# Storing the fit object for later reference
PredictorScalerFit=PredictorScaler.fit(Data)
PredictorScalerFitTest=PredictorScaler.fit(Test)



# Generating the standardized values of X and y
X_train=PredictorScalerFit.transform(Data)
X_test=PredictorScalerFitTest.transform(Test)

In [ ]:
from sklearn import linear_model
from sklearn.linear_model import LinearRegression

lm=LinearRegression()
lm.fit(X_train,Y_train)

y_pred_test=lm.predict(X_test)
y_pred_train=lm.predict(X_train)

In [ ]:
y_pred_test.shape, Y_test.shape

In [ ]:
from sklearn.metrics import r2_score

print('R-sqaure on train data: {}'.format(r2_score(y_true=Y_train, y_pred=y_pred_train)))
print('R-sqaure on test data: {}'.format(r2_score(y_true=Y_test, y_pred=y_pred_test)))

#Standard error/RMSE
error_train=y_pred_train-Y_train
error_test=y_pred_test-Y_test

print('RMSE on train data: {}'.format(((error_train**2).mean())**0.5))
print('RMSE on test data: {}'.format(((error_test**2).mean())**0.5))

In [ ]:
from sklearn.feature_selection import RFE
import statsmodels.api as sm

n_features_list = list(range(4, 31)) #checking for optimal number of features between 4 to 30
train_adjusted_r2 = []
train_r2 = []
test_r2 = []
train_RMSE=[]
test_RMSE=[]

for n_features in range(4, 31):

    # RFE with n features
    lm = LinearRegression()

    # specifying number of features
    rfe_n = RFE(estimator=lm, n_features_to_select=n_features)

    # fit with n features
    rfe_n.fit(X_train, Y_train)

    # selecting features selected by rfe_n
    col_n = X_train.columns[rfe_n.support_] #rfe_n.support_: returns an array with boolean values to indicate whether 
    #an attribute was selected using RFE

    # training & test data for n selected columns
    X_train_rfe_n = X_train[col_n]
    X_test_rfe_n = X_test[col_n]


    # add a constant to the model
    X_train_rfe_n = sm.add_constant(X_train_rfe_n)


    X_test_rfe_n = sm.add_constant(X_test_rfe_n, has_constant='add')

    
    
    # fitting the model with n featues
    lm_n = sm.OLS(y_train, X_train_rfe_n).fit()
    
    
    # # Making predictions
    y_pred_test = lm_n.predict(X_test_rfe_n)
    y_pred_train = lm_n.predict(X_train_rfe_n)
    
    
    #Calculating evaluation metrics
    
    #R-square
    train_adjusted_r2.append(lm_n.rsquared_adj)
    train_r2.append(lm_n.rsquared)
    test_r2.append(r2_score(Y_test, y_pred_test))
    
    #RMSE/stan. error
    error_test=y_pred_test-Y_test
    error_train=y_pred_train-Y_train
    
    test_RMSE.append(((error_test**2).mean())**0.5)
    train_RMSE.append(((error_train**2).mean())**0.5)
    